# Synthetic Research Workshop

Run `make research-demo` from the repository root before opening this notebook. The records are entirely invented. This notebook shows aggregate results only and does not establish clinical findings.


In [ ]:
from src.core.config import load_config
from src.core.db import connect

config = load_config('configs/research_synthetic.duckdb.yaml')


## Row grain and cohort flow

Patients, admissions, and ICU stays have different row grains. The modeling table chooses one index admission per patient.


In [ ]:
with connect(config) as con:
    counts = con.execute('''
        SELECT
          (SELECT count(*) FROM silver_patients) AS patients,
          (SELECT count(*) FROM silver_admissions) AS admissions,
          (SELECT count(*) FROM silver_icustays) AS icu_stays,
          (SELECT count(*) FROM gold_research_index_admissions) AS index_admissions,
          (SELECT count(DISTINCT subject_id) FROM gold_research_index_admissions) AS index_patients
    ''').fetchdf()
counts


## Aggregate exploratory view

The class counts below come from invented diagnosis codes. The chart is a visualization of synthetic cohort composition.


In [ ]:
import plotly.express as px

with connect(config) as con:
    disease_counts = con.execute('''
        SELECT coalesce(primary_disease_class, 'unmapped') AS disease_class,
               count(*) AS index_admissions
        FROM gold_research_index_admissions
        GROUP BY disease_class ORDER BY disease_class
    ''').fetchdf()
px.bar(disease_counts, x='disease_class', y='index_admissions',
       title='Synthetic index admissions by primary class').show()


## Safe aggregation before a one-to-many join

First count lab events per admission, then join those counts to admissions. Joining raw labs to raw charts first would multiply event rows.


In [ ]:
with connect(config) as con:
    lab_summary = con.execute('''
        WITH labs_per_admission AS (
          SELECT hadm_id, count(*) AS lab_events
          FROM silver_labs GROUP BY hadm_id
        )
        SELECT a.admission_type, count(*) AS admissions,
               avg(coalesce(l.lab_events, 0)) AS mean_lab_events
        FROM silver_admissions a
        LEFT JOIN labs_per_admission l ON a.hadm_id = l.hadm_id
        GROUP BY a.admission_type ORDER BY admissions DESC
    ''').fetchdf()
lab_summary


## Observation window and outcomes

The prediction time is 24 hours after ICU entry (or hospital admission without ICU). LOS and death occur later. Readmission is only defined for survivors.


In [ ]:
with connect(config) as con:
    outcomes = con.execute('''
        SELECT count(*) AS index_admissions,
               sum(CASE WHEN prediction_time <= index_dischtime THEN 1 ELSE 0 END) AS complete_windows,
               avg(hospital_los_days) AS mean_hospital_los_days,
               avg(in_hospital_mortality_flag) AS mortality_rate,
               avg(urgent_readmitted_30d) AS survivor_readmission_rate
        FROM gold_research_index_admissions
    ''').fetchdf()
outcomes


## Model comparisons

The generated aggregate report compares dummy, linear, and forest models; uses validation data to select a model; and reports test metrics, calibration, subgroup checks, and permutation importance. No patient-level predictions are exported.


In [ ]:
from IPython.display import Markdown, display

summary_path = config.outputs_public_path / 'aggregate_results' / 'research_demo_summary.md'
display(Markdown(summary_path.read_text()))
